# PS05 — Integrated Solution and Reproducibility Notebook

**Course:** Conversational AI, WS 2026/27  
**Purpose:** Run the four supplied components, record the required HNSW benchmark, inspect hybrid retrieval, evaluate NotebookLM evidence, and produce a provenance/safety demonstration.

> Run this notebook from the same directory as the four `.py` files and `audit_evidence_template.json`. The NotebookLM evaluator requires real observations entered into the JSON evidence file. It deliberately does not fabricate NotebookLM results.

## 0. Environment and project files

Required core dependency: `numpy`. Optional semantic injection screening: `sentence-transformers`. The first embedding-model run may download model weights.

The HNSW benchmark is intentionally an explicit cell because the assignment-sized run (25,000 × 768) can take substantial time with a pure-Python graph implementation.

## 1. HNSW — correctness smoke tests

The supplied implementation uses cosine distance, probabilistic layer assignment, greedy descent, and best-first layer search. Its neighbour selection is a simplified nearest-distance pruning strategy, not the full diversity heuristic of production HNSW.

In [1]:
%run 01_HNSW_Vector_Index.py

Smoke tests passed.


### 1.1 Assignment-scale benchmark: 25,000 vectors × 768 dimensions

Run this cell for the required benchmark. It measures build time, mean query latency, exact flat-search latency, speedup, and recall@10 on the same generated dataset. Record the actual output in the report; do not substitute the earlier 5,000 × 128 result.

In [2]:
RUN_FULL_HNSW_BENCHMARK = True  # Change to True to run the assignment-scale benchmark.
if RUN_FULL_HNSW_BENCHMARK:
    hnsw_assignment_result = run_latency_benchmark(
        N=25_000, dim=768, num_queries=10, seed=42
    )
else:
    print("Skipped. Set RUN_FULL_HNSW_BENCHMARK=True to run N=25,000, dim=768.")

=== HNSW vs. exact flat search (N=25000, dim=768) ===


Indexed vectors:             25000
Build time:                  68.45 s
Mean HNSW query latency:     5.096 ms
Mean exact flat latency:     5.311 ms
Measured speedup:            1.04x
Mean recall@10:              0.250
Note: these measurements apply only to this run and configuration.


## 2. BM25 and Reciprocal Rank Fusion

The component demonstration uses the assignment parameters `k1=1.5`, `b=0.75`, and `RRF k=60`. The dense ranking supplied to the demonstration is illustrative input, not an embedding benchmark.

In [3]:
%run 02_BM25_and_RRF_Combiner.py

BM25/RRF smoke tests passed.

=== BM25 + RRF component demonstration ===
Parameters: k1=1.5, b=0.75, RRF k=60

Exact identifier lookup: SERIAL_NO_AURA_9982X
BM25 ranking: ['DOC_003', 'DOC_001', 'DOC_002', 'DOC_004']
Supplied dense ranking: ['DOC_001', 'DOC_002', 'DOC_004', 'DOC_003']
Hybrid RRF ranking: [('DOC_001', 0.03252247488101534), ('DOC_003', 0.032018442622950824), ('DOC_002', 0.03200204813108039), ('DOC_004', 0.03149801587301587)]

Conceptual retrieval: GraphRAG knowledge graph synthesis
BM25 ranking: ['DOC_004', 'DOC_001', 'DOC_002', 'DOC_003']
Supplied dense ranking: ['DOC_004', 'DOC_002', 'DOC_001', 'DOC_003']
Hybrid RRF ranking: [('DOC_004', 0.03278688524590164), ('DOC_001', 0.03200204813108039), ('DOC_002', 0.03200204813108039), ('DOC_003', 0.03125)]

[PASS] BM25 and RRF component demonstration completed.
Note: dense rankings above are illustrative inputs, not measured embeddings.


### 2.1 Reproducibility note

For the exact identifier example, distinguish the lexical BM25 winner from the hybrid RRF winner. In the uploaded script's recorded demonstration, BM25 ranks `DOC_003` first, while hybrid RRF ranks `DOC_001` first (score approximately 0.0325225).

## 3. NotebookLM grounding and Audio Overview audit

`03_NotebookLM_RAG_Audit.py` evaluates the manually recorded `audit_evidence_template.json`; it does not connect to or operate NotebookLM. The current record contains three source records, five grounded queries, 22 citation records, three negative probes, and Audio Overview metadata. The recorded audio artifact and duration (4,258 seconds / 1 hr 10 min 58 sec) were confirmed by the user from the NotebookLM Studio panel; the user also reports assigning 5/5 on each of the three audio rubric dimensions.

**Interpretation:** the evaluator's `EVIDENCE RECORDS COMPLETE` status means required fields are populated. It does not independently validate claim-to-passage entailment, refusal quality, scoring, or line-by-line transcript alignment. The citation and refusal ratios below are the judgments recorded in the evidence JSON and remain subject to independent human review.


In [11]:
import subprocess, sys
audit_run = subprocess.run(
    [sys.executable, '03_NotebookLM_RAG_Audit.py', 'audit_evidence_template.json'],
    text=True, capture_output=True
)
print(audit_run.stdout)
if audit_run.stderr:
    print(audit_run.stderr)
print('Evaluator exit code:', audit_run.returncode, '(1 means evidence is still incomplete)')


=== PS05 NotebookLM RAG Audit: Evidence Evaluation ===
{
  "source_records": 3,
  "grounded_queries_recorded": 5,
  "citations_recorded": 22,
  "entailed_citations_verified": 22,
  "citation_precision": 1.0,
  "citation_badges_verified": 22,
  "highlighted_spans_recorded": 22,
  "negative_probes_recorded": 3,
  "proper_refusals_verified": 3,
  "epistemic_refusal_rate": 1.0,
  "audio_average_score": 5.0,
  "audio_duration_seconds": 4258
}

Evidence checks: 1

Audit status: EVIDENCE RECORDS COMPLETE.
The evaluator checks recorded evidence; human review is still required for entailment and scoring.

Evaluator exit code: 0 (1 means evidence is still incomplete)


The evidence record is populated from the user's reported NotebookLM observations. Before final submission, independently review each citation badge and highlighted passage against its claim, inspect the three negative-probe responses, and compare the transcript line by line with the rendered audio. Update the JSON only when the review supports a change. Then rerun this evaluator and save the executed notebook.


## 4. Prompt-injection screening, RAG Triad, and provenance

The script runs heuristic screening and attempts embedding-based screening using Sentence Transformers. The lexical RAG Triad scores are proxies, not proof of semantic entailment. A clean example only shows that the example chunk fell below the configured threshold; it does not prove that the embedding detector catches novel paraphrased attacks.

In [5]:
%run 04_RAG_Provenance_and_Safety.py

Safety/RAG Triad/provenance smoke tests passed.
Heuristic malicious-input test: BLOCK_HEURISTIC
Embedding scan availability: UNAVAILABLE
NOTE: embedding screening was not executed; install sentence-transformers and load the model.

RAG Triad lexical proxy scores:
{
  "context_relevance": 1.0,
  "faithfulness": 1.0,
  "answer_relevance": 1.0,
  "method": "lexical_overlap_proxy",
  "interpretation": "Heuristic proxy scores; they do not establish semantic entailment or replace human review."
}

Example provenance log:
{
  "turn_id": "TURN_001",
  "query": "astrogation telemetry",
  "retrieved_chunks": [
    {
      "chunk_id": "CHUNK_001",
      "similarity_score": 0.8731,
      "text": "Subsystem 05 uses dual-path memory for astrogation telemetry."
    }
  ],
  "injection_scan": {
    "overall_status": "HEURISTIC_ONLY_OR_INCOMPLETE",
    "per_chunk": [
      {
        "chunk_id": "CHUNK_001",
        "status": "HEURISTIC_ONLY",
        "heuristic_matches": [],
        "embedding_status":

## 5. Submission checklist

- [ ] HNSW implementation and smoke tests pass.
- [ ] Assignment-scale HNSW benchmark run and actual metrics recorded (25,000 × 768).
- [ ] BM25/RRF smoke tests pass; hybrid ranking is correctly reported.
- [ ] NotebookLM evidence is populated from actual NotebookLM UI outputs and evaluated.
- [ ] Audio Overview is at least 2 minutes and transcript is verbatim.
- [ ] Heuristic and embedding detector are tested separately, including a paraphrased injection that is not caught by the regex patterns.
- [ ] Provenance includes chunk IDs, similarity scores, scan results, and grounded answer.
- [ ] RAG Triad proxy scores are labelled as proxies.
- [ ] All required files are saved under `weeks/week-05/` using the assignment's exact filenames.